In [1]:
from pyspark.sql import SparkSession
import pyspark
print(pyspark.__version__)
from pyspark.sql import SparkSession
from pyspark.sql.utils import AnalysisException
from pyspark.sql import functions as F

3.4.1


In [3]:
# spark.stop()

In [2]:
spark = (
    SparkSession.builder 
    .appName("preprocess_forex") 
    .master("spark://spark-master:7077") 
    .config("spark.cores.max", "1")
    .config("spark.executor.cores", "1")
    .enableHiveSupport()
    .config("hive.metastore.uris", "thrift://hive-metastore:9083")
    .config("spark.hadoop.hive.exec.dynamic.partition", "true")
    .config("spark.hadoop.hive.exec.dynamic.partition.mode", "nonstrict")
    .getOrCreate()
    )

spark.sql("USE DATABASE CryptoPredictions")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/24 14:44:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/11/24 14:44:52 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/11/24 14:44:58 WARN HiveClientImpl: Detected HiveConf hive.execution.engine is 'tez' and will be reset to 'mr' to disable useless hive logic


DataFrame[]

In [4]:
df = spark.read.parquet("hdfs://namenode:8020/nifi/forex")
df.show(3, vertical=True, truncate=False)

[Stage 1:>                                                          (0 + 1) / 1]

-RECORD 0--------------------------------------------------------------------------------------------------------
 ticker       | C:USDAED                                                                                         
 queryCount   | 1                                                                                                
 resultsCount | 1                                                                                                
 adjusted     | true                                                                                             
 results      | [{4, 3.6729, 3.6728, 3.6729, 3.6729, 3.6728, 1763683200000, 4}]                                  
 status       | OK                                                                                               
 request_id   | 03115734e56e15ff780591f3892f9336                                                                 
 count        | 1                                                                       

In [5]:
def unpack_results(df, results_col='results', ticker_col='ticker'):
    """
    Rozpakowuje kolumnę zagnieżdżonych wyników giełdowych i tworzy z niej 
    kolumny analityczne w DataFrame Spark.

    Parametry:
    ----------
    df : pyspark.sql.DataFrame
        DataFrame zawierający kolumnę z wynikami (np. z API finansowego).
    results_col : str, opcjonalnie
        Nazwa kolumny zawierającej zagnieżdżone wyniki (default 'results').
    ticker_col : str, opcjonalnie
        Nazwa kolumny z symbolami instrumentów finansowych (default 'ticker').

    Zwraca:
    -------
    pyspark.sql.DataFrame
        DataFrame z rozpakowanymi kolumnami: VolumeTraded, OpeningPrice,
        HighestDailyPrice, LowestDailyPrice, ClosingPrice, AveragedPrice,
        NumberOfTrades, Date, PartitionDate, CurrencyTo.
        Niepotrzebne kolumny źródłowe są usunięte.
    """

    # Rozpakowanie pierwszego elementu z listy wyników
    df = df.withColumn("res", F.col(results_col).getItem(0))

    # Tworzenie osobnych kolumn z wartościami giełdowymi
    df = df.withColumn("VolumeTraded", F.col("res.v")) \
           .withColumn("OpeningPrice", F.col("res.o")) \
           .withColumn("HighestDailyPrice", F.col("res.h")) \
           .withColumn("LowestDailyPrice", F.col("res.l")) \
           .withColumn("ClosingPrice", F.col("res.c")) \
           .withColumn("AveragedPrice", F.col("res.vw")) \
           .withColumn("timestamp", F.col("res.t")) \
           .withColumn("NumberOfTrades", F.col("res.n"))

    # Czyszczenie kolumny ticker i wyodrębnienie walut
    df = df.withColumn("ticker_clean", F.regexp_replace(F.col(ticker_col), "^C:", "")) \
           .withColumn("CurrencyFrom", F.col("ticker_clean").substr(1, 3)) \
           .withColumn("CurrencyTo", F.expr("substring(ticker_clean, 4, length(ticker_clean))"))

    # Konwersja timestamp na datę i stworzenie kolumny do partycjonowania
    df = df.withColumn("Date", F.to_date(F.from_unixtime(F.col("timestamp") / 1000))) \
           .withColumn("PartitionDate", F.col("Date"))

    # Usunięcie kolumn pomocniczych i zbędnych
    df = df.drop('results', 'res', 'ticker_clean', 'ticker', 
                 'queryCount', 'resultsCount', 'adjusted', 
                 'status', 'request_id', 'count', 'timestamp', 'CurrencyFrom')

    return df

In [6]:
unpacked = unpack_results(df)
unpacked = unpacked.where(unpacked.PartitionDate.isNotNull()) # Usuwamy nulle
unpacked.show(3,vertical=True, truncate=False)

-RECORD 0------------------------------
 VolumeTraded      | 4                 
 OpeningPrice      | 3.6728            
 HighestDailyPrice | 3.6729            
 LowestDailyPrice  | 3.6728            
 ClosingPrice      | 3.6729            
 AveragedPrice     | 3.6729            
 NumberOfTrades    | 4                 
 CurrencyTo        | AED               
 Date              | 2025-11-21        
 PartitionDate     | 2025-11-21        
-RECORD 1------------------------------
 VolumeTraded      | 258746            
 OpeningPrice      | 1.55051           
 HighestDailyPrice | 1.557268550961613 
 LowestDailyPrice  | 1.547269070091289 
 ClosingPrice      | 1.5487            
 AveragedPrice     | 1.5515            
 NumberOfTrades    | 258746            
 CurrencyTo        | AUD               
 Date              | 2025-11-21        
 PartitionDate     | 2025-11-21        
-RECORD 2------------------------------
 VolumeTraded      | 5429              
 OpeningPrice      | 1424.5135         


In [7]:
(unpacked.write
  .mode("append")
  .format("hive")
  .partitionBy("PartitionDate")
  .saveAsTable("USDExchangeRates"))

25/11/24 14:46:00 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.
                                                                                

In [15]:
spark.sql("""
    SELECT * FROM USDExchangeRates
        """).show(5)

+----------+----------+--------------+------------+-----------------+-----------------+-----------------+-----------------+-------------+-------------+
|CurrencyTo|      Date|NumberOfTrades|VolumeTraded|     OpeningPrice|     ClosingPrice| LowestDailyPrice|HighestDailyPrice|AveragedPrice|PartitionDate|
+----------+----------+--------------+------------+-----------------+-----------------+-----------------+-----------------+-------------+-------------+
|       AED|2025-11-17|             6|         6.0|           3.6728|           3.6715|           3.6715|           3.6729|       3.6726|   2025-11-17|
|       AUD|2025-11-17|        213932|    213932.0|1.530690341343946|1.539408866995074|1.529051987767584|1.542709924252943|       1.5354|   2025-11-17|
|       ALL|2025-11-17|             5|         5.0| 81.1614181708525|            83.03|  81.152249895489|            83.03|      82.2807|   2025-11-17|
|       ARS|2025-11-17|          4644|      4644.0|        1409.3161|        1386.9951| 

In [14]:
spark.sql("""
    SELECT DATE, COUNT(*) FROM USDExchangeRates
    GROUP BY DATE
    ORDER BY DATE
        """).show()

+----------+--------+
|      DATE|count(1)|
+----------+--------+
|2025-11-14|     120|
|2025-11-16|     121|
|2025-11-17|     121|
|2025-11-19|     118|
|2025-11-20|     110|
|2025-11-21|     119|
|2025-11-23|     119|
+----------+--------+



In [11]:
spark.stop()